In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ==========================================
# 1. INSTALL LIBRARIES
# ==========================================
!pip install -q transformers datasets torch scikit-learn accelerate sentencepiece

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EvalPrediction
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. CONFIGURATION
# ==========================================
MODEL_ID = "tasksource/ModernBERT-large-nli"

# High context length for interview answers
MAX_LENGTH = 2048

# Training Hyperparameters (Optimized for A100/High VRAM)
BATCH_SIZE = 2
GRAD_ACCUMULATION = 8   # Effective batch size = 16
LEARNING_RATE = 1e-5    # Low LR for fine-tuning large models
EPOCHS = 10             # Increased epochs as Focal Loss can take longer to converge

# Labels
LABEL_MAP = {'Clear Reply': 0, 'Ambivalent': 1, 'Clear Non-Reply': 2}
ID2LABEL = {0: 'Clear Reply', 1: 'Ambivalent', 2: 'Clear Non-Reply'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(LABEL_MAP)

# ==========================================
# 3. DATA PREPARATION & CLASS WEIGHTS
# ==========================================
print("--- Loading Data ---")
dataset = load_dataset("ailsntua/QEvasion")

# --- Calculate Class Weights (Ported from best-sofar.ipynb) ---
def get_class_weights(dataset, label_map, device):
    # Extract labels from the dataset
    # We need to map string labels to integers first to count them
    train_labels = [label_map[label] for label in dataset["train"]["clarity_label"]]

    label_counts = Counter(train_labels)
    total_samples = len(train_labels)
    num_classes = len(label_map)

    # Calculate class weights (inverse frequency)
    class_weights = []
    for i in range(num_classes):
        count = label_counts.get(i, 0)
        if count == 0:
            weight = 0 # Handle potential zero division if a class is missing
        else:
            weight = total_samples / (num_classes * count)
        class_weights.append(weight)

    # Convert to tensor and move to device
    return torch.tensor(class_weights, dtype=torch.float32).to(device)

class_weights = get_class_weights(dataset, LABEL_MAP, device)
print(f"Calculated Class Weights: {class_weights}")

# --- Tokenization ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.model_max_length = MAX_LENGTH

def preprocess_function(examples):
    # NLI Formulation:
    # Premise = Interview Answer
    # Hypothesis = "The speaker explicitly answers the question: {Question}"
    hypotheses = [f"The speaker explicitly answers the question: {q}" for q in examples['question']]
    premises = examples['interview_answer']

    model_inputs = tokenizer(
        premises,
        hypotheses,
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False
    )

    # Map string labels to integers
    if 'clarity_label' in examples:
        model_inputs["labels"] = [LABEL_MAP[label] for label in examples['clarity_label']]

    return model_inputs

encoded_dataset = dataset.map(preprocess_function, batched=True)

# ==========================================
# 4. CUSTOM LOSS & TRAINER
# ==========================================

# Focal Loss Implementation (Ported from best-sofar.ipynb)
class FocalLoss(nn.Module):
    """
    Multi-class Focal loss implementation
    Focal loss helps address class imbalance by focusing on hard examples
    """
    def __init__(self, gamma=2.0, weight=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, input, target):
        # Calculate cross entropy loss
        ce_loss = F.cross_entropy(input, target, reduction='none', weight=self.weight, ignore_index=self.ignore_index)

        # Get probabilities
        pt = torch.exp(-ce_loss)

        # Compute focal loss
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()

# Custom Trainer (Ported from best-sofar.ipynb)
class CustomTrainer(Trainer):
    """
    Custom trainer that uses Focal Loss with class weights
    """
    def __init__(self, *args, class_weights=None, focal_gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_loss = FocalLoss(gamma=focal_gamma, weight=class_weights)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Ensure weights are on the same device as logits
        if self.class_weights is not None:
            self.class_weights = self.class_weights.to(logits.device)
            self.focal_loss.weight = self.class_weights

        loss = self.focal_loss(logits, labels)

        return (loss, outputs) if return_outputs else loss

# ==========================================
# 5. MODEL & METRICS Setup
# ==========================================
config = AutoConfig.from_pretrained(MODEL_ID)
config.max_position_embeddings = MAX_LENGTH
config.num_labels = NUM_LABELS
config.id2label = ID2LABEL
config.label2id = LABEL2ID
config.use_cache = False # Disable cache for gradient checkpointing compatibility

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    config=config,
    ignore_mismatched_sizes=True
)
model.to(device)

def compute_metrics(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

# ==========================================
# 6. TRAINING
# ==========================================
training_args = TrainingArguments(
    output_dir="./tasksource/ModernBERT-large-nli",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,

    # Evaluation Strategy
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # Optimizations
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    logging_steps=25,
    report_to="none"
)

# Initialize CustomTrainer with Class Weights
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    class_weights=class_weights, # Pass weights here
    focal_gamma=2.0              # Gamma for Focal Loss
)

print("\n--- Starting Training (NLI + Focal Loss + Class Weights) ---")
trainer.train()

# ==========================================
# 7. FINAL EVALUATION
# ==========================================
print("\n--- Final Evaluation on Test Set ---")
predictions = trainer.predict(encoded_dataset["test"])
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("\nClassification Report:")
print(classification_report(labels, preds, target_names=list(ID2LABEL.values())))

print("\nConfusion Matrix:")
print(confusion_matrix(labels, preds))

# Save the model
save_path = "./deberta_nli_best_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

Using device: cuda
--- Loading Data ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

...

/tmp/ipython-input-3777673555.py:140: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.



--- Starting Training (NLI + Focal Loss + Class Weights) ---


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,3.080700,0.256415,0.386364,0.384374,0.325507
2,1.217800,0.362780,0.509740,0.525765,0.505397
3,0.475400,0.480249,0.688312,0.631477,0.700364
4,0.403900,0.969908,0.759740,0.665801,0.756498
5,0.202300,0.948400,0.737013,0.655358,0.732971
6,0.112600,1.336368,0.737013,0.650244,0.737498
7,0.053100,1.231949,0.743506,0.668635,0.737574
8,0.020100,1.291679,0.750000,0.680886,0.750475
9,0.016000,1.239017,0.753247,0.700970,0.754988
10,0.007900,1.255257,0.753247,0.698187,0.753553



--- Final Evaluation on Test Set ---



Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.58      0.62      0.60        79
     Ambivalent       0.83      0.82      0.82       206
Clear Non-Reply       0.71      0.65      0.68        23

       accuracy                           0.75       308
      macro avg       0.71      0.70      0.70       308
   weighted avg       0.76      0.75      0.75       308


Confusion Matrix:
[[ 49  28   2]
 [ 34 168   4]
 [  2   6  15]]
Model saved to ./deberta_nli_best_model


In [3]:
# ==========================================
# CONDITIONAL SAVING TO GOOGLE DRIVE
# ==========================================

# 1. Define where you want to save the model in your Drive
drive_save_path = "/content/drive/MyDrive/DeBERTa_NLI_Focal_Best"

# 2. Get the evaluation metrics on the test set
print("Evaluating model to check F1 score...")
test_results = trainer.evaluate()

# 3. Extract the metric (Trainer adds 'eval_' prefix)
# We try 'eval_f1_macro' first, defaulting to 0 if not found
final_f1_score = test_results.get('eval_f1_macro', 0)
target_threshold = 0.70

# 4. Print Status
print(f"\n========================================")
print(f"Achieved F1 Macro: {final_f1_score:.4f}")
print(f"Target Threshold:  {target_threshold}")
print(f"========================================\n")

# 5. Conditional Save Logic
if final_f1_score >= target_threshold:
    print("SUCCESS: Threshold met. Saving model to Google Drive...")

    # Create the directory if it doesn't exist
    if not os.path.exists(drive_save_path):
        os.makedirs(drive_save_path)

    # Save the model and tokenizer
    trainer.save_model(drive_save_path)
    tokenizer.save_pretrained(drive_save_path)

    print(f"Model and tokenizer successfully saved to: {drive_save_path}")

else:
    print(f"SKIP: Score {final_f1_score:.4f} did not meet the requirement of {target_threshold}.")
    print("Model was NOT saved to Drive (saved locally to runtime only).")

Evaluating model to check F1 score...



Achieved F1 Macro: 0.7010
Target Threshold:  0.7

SUCCESS: Threshold met. Saving model to Google Drive...
Model and tokenizer successfully saved to: /content/drive/MyDrive/DeBERTa_NLI_Focal_Best


In [4]:
from google.colab import runtime

runtime.unassign()